In [ ]:
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json, sys, subprocess
WORK = Path('/kaggle/working/render-kaggle')

def extract(name, expected, target, manifest_name):
    matches = list(Path('/kaggle/input').rglob(name))
    assert len(matches) == 1, 'Attach exactly one private input with ' + name
    source = matches[0]
    assert sha256(source.read_bytes()).hexdigest() == expected, 'Bundle hash differs'
    assert not target.exists(), 'Use a fresh notebook session; do not replace running source'
    with ZipFile(source) as archive:
        manifest = json.loads(archive.read(manifest_name))
        names = archive.namelist()
        assert len(names) == len(set(names)) and set(names) == set(manifest['files']) | {manifest_name}
        assert archive.testzip() is None
        for name in names:
            destination = (target / name).resolve()
            assert ':' not in name and chr(92) not in name and destination.is_relative_to(target.resolve())
            raw = archive.read(name)
            if name != manifest_name:
                assert sha256(raw).hexdigest() == manifest['files'][name], 'Member hash differs'
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(raw)
extract('capstone_train002_20260928_v1.bin', 'f0d1fed52e00b6c5a1da2cade3731237675cdebbfc0b2cde3b2dbda99a4019ac', WORK / 'gpu-src', 'gate3_bundle.json')
extract('render_kaggle_cpu_20261009.bin', 'f391050851519e9cc1080830b615a6cc457b6dd1e21fde156522bb959fad9ede', WORK / 'cpu-src', 'demo_bundle.json')
print('Private bundles verified. Enable Internet, select T4 GPU, and run Cell 2.')


In [ ]:
# Starts one local GPU worker, one authenticated API facade and one public API tunnel.
# Keys come from Kaggle Secrets, never notebook source.
subprocess.run([sys.executable, str(WORK / 'cpu-src/scripts/render_kaggle_bootstrap.py')], check=True)


In [ ]:
record = json.loads((WORK / 'demo-session.json').read_text())
print(json.dumps(record, indent=2))
# This is readiness evidence only. No AI generation is started by this cell.
